# 09 — srcML-DKT

Implementação do srcML-DKT (Pankiewicz, Shi & Baker, EDM 2025) como 4º modelo do TCC 1.

**Arquitetura:** idêntica ao Code-DKT (`CodeDKTModel`) — apenas o extrator de paths AST muda de `javalang` para `srcML` CLI.

**Decisão de comparação justa:** treino em `sequences_code_dkt.pkl` (Run.Program + Compile.Error), avaliação no test set de `sequences_bkt_dkt.pkl` (só Run.Program — mesmo test set dos outros 3 modelos).

**Referência:** Pankiewicz, Shi & Baker (2025). *srcML-DKT*, EDM 2025.

## Seção 1 — Setup

In [1]:
import os
import pickle
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

# Adiciona raiz do projeto ao path
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SEED = 42

def set_global_seed(seed: int) -> None:
    """Seed global para reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CPU count: {os.cpu_count()}")

DATA_DIR = ROOT / "data" / "CSEDM"
RESULTS_DIR = ROOT / "results"
CACHE_PATH = RESULTS_DIR / "srcml_features_cache.pkl"
OUTPUT_PATH = RESULTS_DIR / "srcml_dkt_results_multirun.pkl"

ASSIGNMENT_IDS = [439, 487, 492, 494, 502]
SEEDS = list(range(42, 52))  # 10 runs: seeds 42-51

# Hiperparâmetros: BEST_CDKT_CONFIG (consistência interna com Code-DKT)
BEST_CDKT_CONFIG = {
    "hidden_dim": 200,
    "dropout": 0.1,
    "lr": 0.0005,
    "batch_size": 128,
    "epochs": 40,
    "max_len": 50,
    "R": 50,
}
print("Config:", BEST_CDKT_CONFIG)

Device: cuda
CPU count: 16
Config: {'hidden_dim': 200, 'dropout': 0.1, 'lr': 0.0005, 'batch_size': 128, 'epochs': 40, 'max_len': 50, 'R': 50}


## Seção 2 — Carregar dados

In [2]:
from src.srcml_features import load_code_states

# Sequências de treino: com Compile.Error (fiel ao paper Pankiewicz et al.)
with open(RESULTS_DIR / "sequences_code_dkt.pkl", "rb") as f:
    seqs_train_full = pickle.load(f)

# Sequências de teste: só Run.Program (comparação justa com BKT/DKT/Code-DKT)
with open(RESULTS_DIR / "sequences_bkt_dkt.pkl", "rb") as f:
    seqs_test_full = pickle.load(f)

# CodeStates para lookup de código
code_states = load_code_states(DATA_DIR)
print(f"CodeStates carregados: {len(code_states):,}")

# Verificação de integridade
for aid in ASSIGNMENT_IDS:
    n_train = len(seqs_train_full["train"][aid])
    n_test = len(seqs_test_full["test"][aid])
    print(f"  A{aid}: train={n_train}, test={n_test}")

CodeStates carregados: 69,627
  A439: train=307, test=77
  A487: train=272, test=68
  A492: train=290, test=70
  A494: train=253, test=62
  A502: train=245, test=61


In [3]:
import pandas as pd

# Coletar todos os CodeStateIDs únicos (train + test)
all_csids_train: set[str] = set()
for aid in ASSIGNMENT_IDS:
    for seq in seqs_train_full["train"][aid]:
        all_csids_train.update(seq["events"]["CodeStateID"].astype(str).values)

all_csids_test: set[str] = set()
for aid in ASSIGNMENT_IDS:
    for seq in seqs_test_full["test"][aid]:
        all_csids_test.update(seq["events"]["CodeStateID"].astype(str).values)

all_csids = sorted(all_csids_train | all_csids_test)
print(f"CSIDs únicos em train: {len(all_csids_train):,}")
print(f"CSIDs únicos em test:  {len(all_csids_test):,}")
print(f"CSIDs únicos total:    {len(all_csids):,}")

CSIDs únicos em train: 32,781
CSIDs únicos em test:  10,880
CSIDs únicos total:    43,661


## Seção 3 — Smoke test de extração srcML

Valida o extrator em 100 CodeStateIDs e reporta métricas de transparência:
taxa de parsing, paths por submissão, e cobertura em código não-compilável.

In [4]:
from src.srcml_features import extract_paths_srcml

rng_sample = random.Random(SEED)
sample_csids = rng_sample.sample(all_csids, min(100, len(all_csids)))

t0 = time.time()
n_parsed = 0
n_failed = 0
total_paths = 0

for csid in sample_csids:
    code = code_states.get(csid, "")
    paths = extract_paths_srcml(code)
    if paths:
        n_parsed += 1
        total_paths += len(paths)
    else:
        n_failed += 1

elapsed = time.time() - t0
parse_rate = n_parsed / len(sample_csids) * 100
avg_paths = total_paths / n_parsed if n_parsed > 0 else 0

print(f"Sample size: {len(sample_csids)} CSIDs")
print(f"Parsed com >= 1 path: {n_parsed} ({parse_rate:.1f}%)")
print(f"Falhou (0 paths): {n_failed} ({100 - parse_rate:.1f}%)")
print(f"Paths médios/submissão: {avg_paths:.1f}")
print(f"Tempo total: {elapsed:.2f}s ({elapsed/len(sample_csids)*1000:.1f}ms/sub)")

Sample size: 100 CSIDs
Parsed com >= 1 path: 100 (100.0%)
Falhou (0 paths): 0 (0.0%)
Paths médios/submissão: 48.8
Tempo total: 1.98s (19.8ms/sub)


In [5]:
# Verificar especificamente em Compile.Error events
compile_error_csids = []
for aid in ASSIGNMENT_IDS:
    for seq in seqs_train_full["train"][aid]:
        ev = seq["events"]
        compile_events = ev[ev["EventType"] == "Compile.Error"]
        compile_error_csids.extend(compile_events["CodeStateID"].astype(str).tolist())

compile_error_csids_unique = list(set(compile_error_csids))
sample_ce = rng_sample.sample(compile_error_csids_unique, min(50, len(compile_error_csids_unique)))

n_ce_parsed = sum(
    1 for csid in sample_ce
    if extract_paths_srcml(code_states.get(csid, ""))
)
print(f"Compile.Error CSIDs únicos: {len(compile_error_csids_unique):,}")
print(f"Taxa parsing em sample Compile.Error: {n_ce_parsed}/{len(sample_ce)} ({n_ce_parsed/len(sample_ce)*100:.1f}%)")

Compile.Error CSIDs únicos: 10,855
Taxa parsing em sample Compile.Error: 50/50 (100.0%)


## Seção 4 — Cache completo via build_cache_srcml

Extração paralela de paths para todos os `{len(all_csids):,}` CodeStateIDs únicos.
Usa `multiprocessing.Pool` com `n_workers=os.cpu_count()`. Cache salvo em `results/srcml_features_cache.pkl`.

In [6]:
from src.srcml_features import build_cache_srcml

if CACHE_PATH.exists():
    print(f"Cache já existe em {CACHE_PATH} — carregando...")
    with open(CACHE_PATH, "rb") as f:
        cache_raw = pickle.load(f)
    print(f"Cache carregado: {len(cache_raw):,} entradas")
else:
    n_workers = os.cpu_count()
    print(f"Construindo cache para {len(all_csids):,} CSIDs com {n_workers} workers...")
    t0 = time.time()
    cache_raw = build_cache_srcml(
        code_state_ids=all_csids,
        code_states=code_states,
        n_workers=n_workers,
    )
    elapsed = time.time() - t0
    print(f"Cache construído em {elapsed:.1f}s ({elapsed/60:.1f} min)")
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(cache_raw, f)
    print(f"Cache salvo: {CACHE_PATH}")

Construindo cache para 43,661 CSIDs com 16 workers...


Cache construído em 107.9s (1.8 min)


Cache salvo: /home/leokuntz/Documents/repositories/studies/tcc.edm.kt/results/srcml_features_cache.pkl


In [7]:
# Métricas do cache completo
n_with_paths = sum(1 for v in cache_raw.values() if v)
n_total = len(cache_raw)
parse_rate_full = n_with_paths / n_total * 100
avg_paths_full = np.mean([len(v) for v in cache_raw.values() if v])

print(f"Cache total: {n_total:,} entradas")
print(f"Com >= 1 path: {n_with_paths:,} ({parse_rate_full:.2f}%)")
print(f"Sem paths (falhou): {n_total - n_with_paths:,} ({100 - parse_rate_full:.2f}%)")
print(f"Paths médios/entry: {avg_paths_full:.1f}")
assert parse_rate_full >= 80, f"Taxa de parsing abaixo do critério mínimo: {parse_rate_full:.1f}%"
print(f"\nCritério mínimo 80% parse rate: OK")

Cache total: 43,661 entradas
Com >= 1 path: 43,661 (100.00%)
Sem paths (falhou): 0 (0.00%)
Paths médios/entry: 48.3

Critério mínimo 80% parse rate: OK


## Seção 5 — Vocabulário por assignment

Constrói token_to_idx e path_to_idx a partir do **train set** (`sequences_code_dkt`). Reutiliza `build_vocab` de `code_features.py` — interface idêntica.

In [8]:
from src.code_features import build_vocab

vocabs: dict[int, dict] = {}
problem_to_idxs: dict[int, dict[int, int]] = {}

for aid in ASSIGNMENT_IDS:
    train_seqs = seqs_train_full["train"][aid]
    test_seqs = seqs_test_full["test"][aid]

    # CSIDs do train para vocab (apenas train — sem data leakage)
    train_csids = set()
    for seq in train_seqs:
        train_csids.update(seq["events"]["CodeStateID"].astype(str).values)
    cache_train = {csid: cache_raw.get(csid, []) for csid in train_csids}

    token_to_idx, path_to_idx = build_vocab(cache_train)

    # problem_to_idx: union de train + test (para indexação consistente)
    all_pids = set()
    for seq in train_seqs + test_seqs:
        all_pids.update(int(p) for p in seq["events"]["ProblemID"].values)
    problem_to_idx = {pid: idx for idx, pid in enumerate(sorted(all_pids))}

    vocabs[aid] = {
        "token_to_idx": token_to_idx,
        "path_to_idx": path_to_idx,
        "node_count": len(token_to_idx),
        "path_count": len(path_to_idx),
    }
    problem_to_idxs[aid] = problem_to_idx

    print(f"A{aid}: tokens={len(token_to_idx):,} paths={len(path_to_idx):,} problems={len(problem_to_idx)}")

A439: tokens=569 paths=8,013 problems=10
A487: tokens=952 paths=14,255 problems=10


A492: tokens=1,306 paths=19,639 problems=10


A494: tokens=799 paths=13,684 problems=10
A502: tokens=834 paths=15,310 problems=10


## Seção 6 — Tensorização e smoke train (A439, 5 épocas)

Verifica que o pipeline ponta-a-ponta funciona antes do treino completo.

In [9]:
from src.code_features import build_code_input_tensor
from src.models.code_dkt import CodeDKTModel, train_code_dkt, predict_code_dkt
from src.evaluation import compute_auc

# Smoke test em A439
AID_SMOKE = 439
set_global_seed(SEED)

train_seqs_smoke = seqs_train_full["train"][AID_SMOKE]
test_seqs_smoke = seqs_test_full["test"][AID_SMOKE]
vocab_smoke = vocabs[AID_SMOKE]
p2i_smoke = problem_to_idxs[AID_SMOKE]

# Forward pass com batch de 2 sequências
X_smoke, Y_smoke, mask_smoke = build_code_input_tensor(
    train_seqs_smoke[:2], cache_raw,
    vocab_smoke["token_to_idx"], vocab_smoke["path_to_idx"],
    p2i_smoke, max_len=50, R=50,
)
M_smoke = len(p2i_smoke)
model_smoke = CodeDKTModel(
    input_dim=2 * M_smoke,
    hidden_dim=200,
    output_dim=M_smoke,
    node_count=vocab_smoke["node_count"],
    path_count=vocab_smoke["path_count"],
    dropout=0.1,
    R=50,
).to(device)
with torch.no_grad():
    out_smoke = model_smoke(X_smoke.to(device))
print(f"Forward pass OK — output shape: {out_smoke.shape}  (esperado: [2, 50, {M_smoke}])")
del model_smoke, X_smoke, Y_smoke, mask_smoke, out_smoke

Forward pass OK — output shape: torch.Size([2, 50, 10])  (esperado: [2, 50, 10])


In [10]:
# Smoke train: 5 épocas em A439
set_global_seed(SEED)
smoke_config = {**BEST_CDKT_CONFIG, "epochs": 5}

t0 = time.time()
model_smoke = train_code_dkt(
    train_seqs_smoke, p2i_smoke, vocab_smoke, smoke_config, cache_raw, seed=SEED
)
elapsed_smoke = time.time() - t0
print(f"\nSmoke train OK em {elapsed_smoke:.1f}s")

pred_df_smoke = predict_code_dkt(
    model_smoke, test_seqs_smoke, p2i_smoke, vocab_smoke, cache_raw,
)
auc_smoke = compute_auc(pred_df_smoke, first_attempt_only=True)
print(f"Smoke first-attempt AUC (5 épocas): {auc_smoke*100:.2f}%")
del model_smoke

  Época  1/5 — loss: 0.6707


  Época  2/5 — loss: 0.6052


  Época  3/5 — loss: 0.5463


  Época  4/5 — loss: 0.5115


  Época  5/5 — loss: 0.4873

Smoke train OK em 3.4s


Smoke first-attempt AUC (5 épocas): 59.21%


## Seção 7 — Treino full: 10 runs × 5 assignments × 40 épocas

Usa BEST_CDKT_CONFIG. Treina em `sequences_code_dkt` (com Compile.Error). Avalia no test set de `sequences_bkt_dkt` (só Run.Program).

In [11]:
from src.models.code_dkt import train_and_evaluate

results_all: dict[int, list[dict]] = {aid: [] for aid in ASSIGNMENT_IDS}

t_total = time.time()
for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Seed {seed} ({SEEDS.index(seed)+1}/{len(SEEDS)})")
    print(f"{'='*60}")
    for aid in ASSIGNMENT_IDS:
        print(f"\n  A{aid}...")
        set_global_seed(seed)
        result = train_and_evaluate(
            train_sequences=seqs_train_full["train"][aid],
            test_sequences=seqs_test_full["test"][aid],
            problem_to_idx=problem_to_idxs[aid],
            vocab=vocabs[aid],
            config=BEST_CDKT_CONFIG,
            cache_raw=cache_raw,
            seed=seed,
        )
        results_all[aid].append({
            "seed": seed,
            "all_auc": result["all_auc"],
            "first_auc": result["first_auc"],
            "pred_df": result["pred_df"],
        })
        print(f"    first_auc={result['first_auc']*100:.2f}%  all_auc={result['all_auc']*100:.2f}%")

print(f"\nTreino total: {(time.time()-t_total)/60:.1f} min")


Seed 42 (1/10)

  A439...


  Época  1/40 — loss: 0.6707


  Época  2/40 — loss: 0.6052


  Época  3/40 — loss: 0.5463


  Época  4/40 — loss: 0.5115


...

  Época 37/40 — loss: 0.4137


  Época 38/40 — loss: 0.4119


  Época 39/40 — loss: 0.4076


  Época 40/40 — loss: 0.4061


    first_auc=81.01%  all_auc=72.99%

Treino total: 8.7 min


## Seção 8 — Sumário (mean ± std vs Code-DKT multirun)

In [12]:
# Carregar Code-DKT multirun para comparação
with open(RESULTS_DIR / "code_dkt_results_multirun.pkl", "rb") as f:
    cdkt_ref = pickle.load(f)

print(f"{'Aid':<6} {'srcML first':>14} {'srcML all':>12} {'Code-DKT first':>16} {'Δfirst':>9}")
print("-" * 62)

for aid in ASSIGNMENT_IDS:
    runs = results_all[aid]
    first_aucs = [r["first_auc"] for r in runs]
    all_aucs = [r["all_auc"] for r in runs]

    mean_first = np.mean(first_aucs)
    std_first = np.std(first_aucs)
    mean_all = np.mean(all_aucs)
    std_all = np.std(all_aucs)

    cdkt_first = cdkt_ref[aid]["first_auc_mean"]
    delta = mean_first - cdkt_first

    print(
        f"A{aid:<5} {mean_first*100:.2f}±{std_first*100:.2f}%"
        f"  {mean_all*100:.2f}±{std_all*100:.2f}%"
        f"  {cdkt_first*100:.2f}%"
        f"  {delta*100:+.2f}pp"
    )

Aid       srcML first    srcML all   Code-DKT first    Δfirst
--------------------------------------------------------------
A439   70.41±1.01%  67.25±0.44%  73.27%  -2.86pp
A487   76.56±0.87%  71.80±0.51%  79.56%  -3.00pp
A492   81.93±0.71%  75.80±0.87%  86.12%  -4.19pp
A494   78.30±0.90%  70.04±0.92%  81.85%  -3.54pp
A502   81.17±0.99%  72.59±0.63%  84.98%  -3.82pp


## Seção 9 — Serialização

In [13]:
# Montar pickle final com schema idêntico ao code_dkt_results_multirun.pkl
# Reusa o modelo treinado com seed=42 para model_state_dict_seed42

final_results: dict[int, dict] = {}

for aid in ASSIGNMENT_IDS:
    runs = results_all[aid]
    first_aucs = [r["first_auc"] for r in runs]
    all_aucs = [r["all_auc"] for r in runs]

    # Retreinar com seed=42 para salvar state_dict (model de referência)
    set_global_seed(42)
    result_42 = train_and_evaluate(
        train_sequences=seqs_train_full["train"][aid],
        test_sequences=seqs_test_full["test"][aid],
        problem_to_idx=problem_to_idxs[aid],
        vocab=vocabs[aid],
        config=BEST_CDKT_CONFIG,
        cache_raw=cache_raw,
        seed=42,
    )

    final_results[aid] = {
        "all_auc_mean": float(np.mean(all_aucs)),
        "all_auc_std": float(np.std(all_aucs)),
        "first_auc_mean": float(np.mean(first_aucs)),
        "first_auc_std": float(np.std(first_aucs)),
        "runs": runs,
        "n_train_events": result_42["n_train_events"],
        "n_test_events": result_42["n_test_events"],
        "config": BEST_CDKT_CONFIG,
        "vocab": vocabs[aid],
        "problem_to_idx": problem_to_idxs[aid],
        "model_state_dict_seed42": result_42["model"].state_dict(),
    }
    print(f"A{aid}: first_auc_mean={final_results[aid]['first_auc_mean']*100:.2f}%")

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(final_results, f)
print(f"\nSalvo: {OUTPUT_PATH}")

  Época  1/40 — loss: 0.6707


  Época  2/40 — loss: 0.6052


  Época  3/40 — loss: 0.5463


  Época  4/40 — loss: 0.5115


  Época  5/40 — loss: 0.4873


...

  Época 37/40 — loss: 0.4156


  Época 38/40 — loss: 0.4135


  Época 39/40 — loss: 0.4107


  Época 40/40 — loss: 0.4088


A502: first_auc_mean=81.17%

Salvo: /home/leokuntz/Documents/repositories/studies/tcc.edm.kt/results/srcml_dkt_results_multirun.pkl


## Seção 10 — Sanity checks

In [14]:
# Verificação de schema e coerência
with open(OUTPUT_PATH, "rb") as f:
    r = pickle.load(f)

REQUIRED_KEYS = {
    "all_auc_mean", "all_auc_std", "first_auc_mean", "first_auc_std",
    "runs", "n_train_events", "n_test_events", "config",
    "vocab", "problem_to_idx", "model_state_dict_seed42",
}

errors = []
for aid in ASSIGNMENT_IDS:
    assert aid in r, f"A{aid} ausente no pickle"
    missing = REQUIRED_KEYS - set(r[aid].keys())
    if missing:
        errors.append(f"A{aid}: chaves ausentes {missing}")

    n_runs = len(r[aid]["runs"])
    if n_runs != 10:
        errors.append(f"A{aid}: esperado 10 runs, got {n_runs}")

    seeds = sorted([run["seed"] for run in r[aid]["runs"]])
    if seeds != list(range(42, 52)):
        errors.append(f"A{aid}: seeds incorretos {seeds}")

    run_keys = set(r[aid]["runs"][0].keys())
    expected_run_keys = {"seed", "all_auc", "first_auc", "pred_df"}
    if not expected_run_keys.issubset(run_keys):
        errors.append(f"A{aid}: run keys incorretos {run_keys}")

if errors:
    for e in errors:
        print(f"ERRO: {e}")
    raise AssertionError("Sanity checks falharam")

print("Schema OK — todos os sanity checks passaram")

Schema OK — todos os sanity checks passaram


In [15]:
# Reprodutibilidade: verificar que seed=42 no pickle bate com run da seção 7
for aid in ASSIGNMENT_IDS:
    run_42_from_multirun = next(run for run in r[aid]["runs"] if run["seed"] == 42)
    run_42_from_s9 = next(run for run in results_all[aid] if run["seed"] == 42)

    diff = abs(run_42_from_multirun["first_auc"] - run_42_from_s9["first_auc"])
    if diff > 1e-6:
        print(f"A{aid}: DIVERGÊNCIA seed=42 first_auc diff={diff:.6f}")
    else:
        print(f"A{aid}: seed=42 reprodutível (diff={diff:.2e})")

A439: seed=42 reprodutível (diff=0.00e+00)
A487: seed=42 reprodutível (diff=0.00e+00)
A492: seed=42 reprodutível (diff=0.00e+00)
A494: seed=42 reprodutível (diff=0.00e+00)
A502: seed=42 reprodutível (diff=0.00e+00)


In [16]:
# Comparação final: srcML-DKT vs Code-DKT vs DKT
with open(RESULTS_DIR / "code_dkt_results_multirun.pkl", "rb") as f:
    cdkt_ref = pickle.load(f)
with open(RESULTS_DIR / "dkt_results_multirun.pkl", "rb") as f:
    dkt_ref = pickle.load(f)

print(f"{'Aid':<6} {'srcML':>10} {'Code-DKT':>12} {'DKT':>10} {'Δ(srcML-CDKT)':>15} {'Δ(srcML-DKT)':>14}")
print("-" * 72)
for aid in ASSIGNMENT_IDS:
    sm = r[aid]["first_auc_mean"]
    cd = cdkt_ref[aid]["first_auc_mean"]
    dk = dkt_ref[aid]["first_auc_mean"]
    print(
        f"A{aid:<5} {sm*100:.2f}%"
        f"   {cd*100:.2f}%"
        f"   {dk*100:.2f}%"
        f"   {(sm-cd)*100:+.2f}pp"
        f"   {(sm-dk)*100:+.2f}pp"
    )

Aid         srcML     Code-DKT        DKT   Δ(srcML-CDKT)   Δ(srcML-DKT)
------------------------------------------------------------------------
A439   70.41%   73.27%   75.56%   -2.86pp   -5.15pp
A487   76.56%   79.56%   76.70%   -3.00pp   -0.14pp
A492   81.93%   86.12%   82.05%   -4.19pp   -0.12pp
A494   78.30%   81.85%   80.17%   -3.54pp   -1.87pp
A502   81.17%   84.98%   80.78%   -3.82pp   +0.38pp
